# PM10 — Notebook 2: WAIC/DIC


In [ ]:
# Kaggle: Internet must be ON only if pyro is not already available.
!pip -q install pyro-ppl openpyxl

In [ ]:
import os, glob, pickle
from pathlib import Path
TEST_MODE=False
SEED=1405
MODELS=['proposed_matern_fscsn','gaussian','skew_exponential']
OUT_DIR=Path('/kaggle/working/PM10_model_comparison/full_fit')
OUT_DIR.mkdir(parents=True,exist_ok=True)

#
hits=glob.glob('/kaggle/input/**/pm10_prepared.pkl',recursive=True)+glob.glob('/kaggle/working/**/pm10_prepared.pkl',recursive=True)
if not hits: raise FileNotFoundError('pm10_prepared.pkl را به Dataset این Notebook اضافه کنید.')
PREP_PATH=hits[0]
with open(PREP_PATH,'rb') as f: prep=pickle.load(f)
print(prep['meta'])

In [ ]:
import math, os, time, pickle, json, glob, gc, random
from functools import partial
from pathlib import Path
import numpy as np, pandas as pd
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
from scipy.special import kv, gamma
from scipy.stats import norm

pyro.set_rng_seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_default_dtype(torch.float64)

def log_ndtr(x):
    return torch.special.log_ndtr(x) if hasattr(torch.special,'log_ndtr') else torch.log(torch.clamp(0.5*(1+torch.erf(x/math.sqrt(2))),min=1e-300))

class TorchFCSN(dist.TorchDistribution):
    arg_constraints={}
    support=dist.constraints.real
    has_rsample=False
    def __init__(self,mu,sigma,C,Lambda,validate_args=None):
        self.mu,self.sigma,self.C,self.Lambda=mu,sigma,C,Lambda
        self.dim=mu.shape[-1]
        super().__init__(torch.Size(),torch.Size([self.dim]),validate_args=validate_args)
        L=torch.linalg.cholesky(C)
        C12=L.T
        I=torch.eye(self.dim,dtype=C.dtype,device=C.device)
        C12_inv=torch.linalg.solve_triangular(C12,I,upper=True)
        one=torch.ones(self.dim,dtype=C.dtype,device=C.device)
        delta=Lambda/torch.sqrt(1+Lambda**2)
        bd=math.sqrt(2/math.pi)*delta
        st=sigma/torch.sqrt(1-bd**2+1e-12)
        self.L,self.C12,self.one,self.bd,self.st=L,C12,one,bd,st
        self.mux=mu-(bd*st)*(one@C12)
        self.D=(Lambda/(st+1e-12))*C12_inv
        self.logdet_cov=self.dim*torch.log(st**2+1e-12)+2*torch.log(torch.diag(L)).sum()
        self.c1=st*Lambda/torch.sqrt(1+Lambda**2)
        self.c2=st/torch.sqrt(1+Lambda**2)
    def sample(self,sample_shape=torch.Size()):
        shape=sample_shape+torch.Size([self.dim])
        u1=torch.randn(shape,dtype=self.mu.dtype,device=self.mu.device)
        u2=torch.randn(shape,dtype=self.mu.dtype,device=self.mu.device)
        return self.mux+(self.c1*torch.abs(u2))@self.C12+(self.c2*u1)@self.C12
    def log_prob(self,x):
        if x.ndim==1: x=x.unsqueeze(0)
        diff=x-self.mux
        q=torch.linalg.solve_triangular(self.L,(diff/(self.st+1e-12)).T,upper=False).T
        logg=-0.5*(self.dim*math.log(2*math.pi)+self.logdet_cov+(q*q).sum(-1))
        z=(x-self.mu)@self.D+(self.bd*self.Lambda)*self.one
        return self.dim*math.log(2)+logg+log_ndtr(z).sum(-1)

def corr_matrix(knots,kind,phi=None,nu=1.5,jitter=1e-6):
    D=np.sqrt(((knots[:,None,:]-knots[None,:,:])**2).sum(2))
    if phi is None: phi=np.median(D[D>0])
    r=D/(phi+1e-12)
    if kind=='exponential':
        C=np.exp(-r)
    elif kind=='matern32':
        C=(1+math.sqrt(3)*r)*np.exp(-math.sqrt(3)*r)
    else: raise ValueError(kind)
    return C+jitter*np.eye(len(knots)),float(phi)

def make_tensors(prep,idx,C):
    return tuple(torch.tensor(np.asarray(a)[idx]) for a in [prep['y'],prep['X'],prep['B'],prep['G']])+(torch.tensor(C),)

def model(y,X,B,G,C,model_name):
    N,P=X.shape; K=B.shape[1]; J=G.shape[1]
    sigma_eps=pyro.sample('sigma_eps',dist.HalfNormal(5.0))
    sigma_theta=pyro.sample('sigma_theta',dist.HalfNormal(5.0))
    beta0=pyro.sample('beta0',dist.Normal(0,10))
    beta_rest=pyro.sample('beta_rest',dist.Normal(0,2).expand([P-1]).to_event(1))
    beta=torch.cat([beta0[None],beta_rest])
    zero=torch.zeros(K,dtype=y.dtype)
    L=torch.linalg.cholesky(C)
    if model_name!='gaussian':
        Lambda=pyro.sample('Lambda',dist.HalfNormal(2.0))
    th=[]
    for j in range(J):
        if model_name=='gaussian':
            thj=pyro.sample(f'theta_{j}',dist.MultivariateNormal(zero,scale_tril=sigma_theta*L))
        else:
            thj=pyro.sample(f'theta_{j}',TorchFCSN(zero,sigma_theta,C,Lambda))
        th.append(thj)
    Theta=torch.stack(th)
    mu=X@beta+torch.sum((B@Theta.T)*G,dim=1)
    pyro.sample('y',dist.Normal(mu,sigma_eps).to_event(1),obs=y)

def flatten_samples(samples):
    out={}
    for k,v in samples.items():
        a=v.detach().cpu().numpy()
        out[k]=a.reshape((-1,)+a.shape[2:]) if a.ndim>=2 else a.reshape(-1)
    return out

def posterior_mu(samples,X,B,G,J,draw_ids=None):
    X=np.asarray(X); B=np.asarray(B); G=np.asarray(G)
    S=len(samples['sigma_eps'])
    if draw_ids is None: draw_ids=np.arange(S)
    ans=[]
    for s in draw_ids:
        beta=np.r_[samples['beta0'][s],samples['beta_rest'][s]]
        Theta=np.stack([samples[f'theta_{j}'][s] for j in range(J)])
        ans.append(X@beta+np.sum((B@Theta.T)*G,axis=1))
    return np.asarray(ans)

def fit_one(prep,model_name,train_idx,warmup,draws,chains,target_accept,max_tree_depth,seed):
    kind='matern32' if model_name in ['proposed_matern_fscsn','gaussian'] else 'exponential'
    C,phi=corr_matrix(prep['knots'],kind)
    y,X,B,G,Ct=make_tensors(prep,train_idx,C)
    pyro.clear_param_store(); pyro.set_rng_seed(seed)
    kernel=NUTS(partial(model, model_name=model_name),
                target_accept_prob=target_accept,max_tree_depth=max_tree_depth)
    mcmc_kwargs=dict(warmup_steps=warmup,num_samples=draws,num_chains=chains,disable_progbar=False)
    if chains>1:
        mcmc_kwargs['mp_context']='fork'
    mcmc=MCMC(kernel,**mcmc_kwargs)
    t0=time.time(); mcmc.run(y,X,B,G,Ct); runtime=time.time()-t0
    grouped=mcmc.get_samples(group_by_chain=True)
    return grouped,dict(corr_kind=kind,phi_fixed=phi,runtime_sec=runtime)

In [ ]:
def logmeanexp(a,axis=0):
    m=np.max(a,axis=axis,keepdims=True)
    return np.squeeze(m,axis=axis)+np.log(np.mean(np.exp(a-m),axis=axis))

def metrics_full(prep,flat,max_eval_draws=400):
    S=len(flat['sigma_eps']); use=min(S,max_eval_draws)
    ids=np.linspace(0,S-1,use).astype(int)
    MU=posterior_mu(flat,prep['X'],prep['B'],prep['G'],prep['meta']['J'],ids)
    y=np.asarray(prep['y']); sig=np.asarray(flat['sigma_eps'])[ids]
    LL=-0.5*np.log(2*np.pi*sig[:,None]**2)-0.5*((y[None,:]-MU)/sig[:,None])**2
    lppd=logmeanexp(LL,axis=0).sum()
    pwaic=np.var(LL,axis=0,ddof=1).sum()
    waic=-2*(lppd-pwaic)
    dev=-2*LL.sum(1)
    Dbar=dev.mean()
    mu_bar=MU.mean(0); sig_bar=sig.mean()
    Dhat=-2*np.sum(norm.logpdf(y,loc=mu_bar,scale=sig_bar))
    pD=Dbar-Dhat; dic=Dbar+pD
    pred=MU.mean(0)
    rmse=float(np.sqrt(np.mean((y-pred)**2))); mae=float(np.mean(np.abs(y-pred)))
    # predictive intervals include observation noise
    rng=np.random.default_rng(SEED)
    YREP=MU+rng.normal(size=MU.shape)*sig[:,None]
    lo,hi=np.quantile(YREP,[.05,.95],axis=0)
    cov=float(np.mean((y>=lo)&(y<=hi)))
    return dict(WAIC=float(waic),p_WAIC=float(pwaic),DIC=float(dic),p_D=float(pD),
                RMSE_in=float(rmse),MAE_in=float(mae),coverage90_in=cov,eval_draws=use)

In [ ]:
settings=dict(
    warmup=100 if TEST_MODE else 700,
    draws=100 if TEST_MODE else 1500,
    chains=1 if TEST_MODE else 2,
    target_accept=0.95,
    max_tree_depth=10
)
rows=[]
for m in MODELS:
    print('\n'+'='*80+'\nMODEL:',m)
    samples,meta=fit_one(prep,m,np.arange(len(prep['y'])),seed=SEED+MODELS.index(m),**settings)
    flat=flatten_samples(samples)
    met=metrics_full(prep,flat,max_eval_draws=100 if TEST_MODE else 400)
    row={'model':m,**meta,**met,**settings}
    rows.append(row)
    with open(OUT_DIR/f'full_{m}.pkl','wb') as f:
        pickle.dump({'samples':{k:v.cpu() for k,v in samples.items()},'meta':row},f)
    pd.DataFrame(rows).to_csv(OUT_DIR/'full_model_comparison_partial.csv',index=False)
    print(row)
    del samples,flat; gc.collect()

res=pd.DataFrame(rows).sort_values('WAIC')
res.to_csv(OUT_DIR/'full_model_comparison.csv',index=False)
res.to_excel(OUT_DIR/'full_model_comparison.xlsx',index=False)
display(res)